# ECONVN2026 — Chaos-aware Bayesian LSTM on VN30 (companion demo)

End-to-end walkthrough of the analysis pipeline on one VN30 ticker (VCB).

**What the notebook does**

1. Loads processed daily + 5-minute bars for VCB from `data/`.
2. Computes the three chaos indicators used in the paper: largest Lyapunov exponent (Rosenstein), Bandt-Pompe permutation entropy, and the Gottwald-Melbourne 0-1 $K$-value.
3. Runs the IAAFT (Schreiber-Schmitz 1996) surrogate-data null test to calibrate $K$ against the leptokurtic-noise null flagged by Federico (2020) and Webel (2012).
4. Trains the five-member LSTM ensemble reported in §4.3 and applies pseudo-BMA stacking.
5. Plots the PIT calibration histogram on the test split.

**What the notebook does not do**

- It does not reproduce the full thirty-ticker rolling chaos-indicator run; that one takes hours and is wired in `src/scripts/run_5m_full.py`.
- It does not scrape live market data; we ship the processed parquet files so the demo is offline-reproducible.

**Scope**: ECONVN2026 conference submission. No Q1 journal upgrade is pursued.

## 0. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data"
print("repo:", REPO_ROOT)
print("data:", sorted(p.name for p in DATA_DIR.iterdir()))

## 1. Load VCB daily and 5-minute returns

In [ ]:
daily = pd.read_parquet(DATA_DIR / "VCB_1D_processed.parquet")
intraday = pd.read_parquet(DATA_DIR / "VCB_5m_processed.parquet")

ret_col = "ret" if "ret" in daily.columns else "log_return"
r_daily = daily[ret_col].dropna().to_numpy()
r_5m = intraday[ret_col].dropna().to_numpy()
print(f"daily bars : {r_daily.size:,}")
print(f"5-min bars : {r_5m.size:,}")

## 2. Chaos indicators

Expect the largest Lyapunov exponent $\lambda$ near zero, permutation entropy near one, and the 0-1 $K$-value near one. Near-unity $K$ on leptokurtic returns is a **known failure mode** of the test (Federico 2020), not a chaos finding.

In [ ]:
from src.python.chaos_indicators import chaos01_K, largest_lyapunov, permutation_entropy

x = r_daily.astype(float)
indicators = {
    "lambda_Rosenstein": largest_lyapunov(x, m=5, tau=1),
    "permutation_entropy": permutation_entropy(x, order=4, delay=1),
    "K_01_test": chaos01_K(x, n_c=30, seed=0),
}
for k, v in indicators.items():
    print(f"{k:>22s} = {v:+.4f}")

## 3. IAAFT surrogate-data null test

We generate amplitude-and-spectrum-matched surrogates and recompute each indicator on the null realisations. A one-sided $p$-value is the fraction of surrogates that reach or exceed the observed value. A surviving indicator ($p \le \alpha$) rejects the linear-stochastic null; a failing indicator is consistent with leptokurtic noise.

In [ ]:
from src.python.surrogates import surrogate_null_test

rng = np.random.default_rng(42)
n_surrogates = 99

report = {}
for name, fn in [
    ("K_01_test", lambda y: chaos01_K(y, n_c=30, seed=0)),
    ("permutation_entropy", lambda y: permutation_entropy(y, order=4, delay=1)),
]:
    res = surrogate_null_test(x, indicator_fn=fn, n_surrogates=n_surrogates, n_iter=50, alpha=0.05, rng=rng)
    report[name] = {
        "observed": res["observed"],
        "null_mean": res["null_mean"],
        "null_std": res["null_std"],
        "p_value": res["p_value"],
        "reject_null": bool(res["reject_null"]),
    }
pd.DataFrame(report).T

## 4. Interpretation

Read the IAAFT table. If `reject_null=False` for $K$, then the near-unity $K$ on VCB daily returns **does not** constitute evidence of low-dimensional chaos — it is statistically indistinguishable from a linear-stochastic process that shares VCB's amplitude distribution and power spectrum. This is the null calibration the paper applies in §4.2 and §4.5.

## 5. Five-member LSTM ensemble + pseudo-BMA stacking

Toggle `RUN_LSTM = True` to reproduce the §4.3 MVP result. The training uses the panel-averaged return series bundled at `data/vn30_panel_1D.parquet`.

**Note.** The forecast evaluation in the paper is reported on the panel-averaged target. A per-ticker replication is logged as future work.

In [ ]:
RUN_LSTM = False

if RUN_LSTM:
    import torch

    from src.python.bma import pseudo_bma_stack
    from src.python.lstm_ensemble import train_ensemble_panel_avg

    panel = pd.read_parquet(DATA_DIR / "vn30_panel_1D.parquet")
    ensemble = train_ensemble_panel_avg(panel, n_members=5, epochs=50, seq_len=60, device="cpu")
    weights = pseudo_bma_stack(ensemble["val_logliks"])
    print("stacking weights:", np.round(weights, 3))
    print("test loglik BMA :", ensemble["test_loglik_bma"])
    print("test loglik unif:", ensemble["test_loglik_unif"])
else:
    print("Set RUN_LSTM=True to train the 5-member ensemble on CPU (~10 min).")

## 6. PIT calibration diagnostic

On the MVP run the central 80% predictive interval traps roughly 98.4% of realisations (1.2% below the 10% quantile, 99.6% below the 90% quantile). That is a signature of scale **under-confidence** (predictive distribution too wide), not over-confidence. The paper corrects this interpretation in §4.4 and §5.1.

## 7. How to cite

```bibtex
@inproceedings{lim2026econvn_vn30chaosbma,
  author    = {Lim, Paul (Fong) and Tram, {anonymised}},
  title     = {Chaos-aware Bayesian {LSTM} on Vietnamese {HF} equity data: a null-calibrated audit of the {VN30} basket},
  booktitle = {Proceedings of ECONVN2026},
  year      = {2026},
  address   = {Ho Chi Minh City, Vietnam}
}
```

Companion repo code is released under the MIT license; see `LICENSE`.